In [7]:
from pyspark.sql.functions import lit, when, col
from functools import reduce
from pyspark.sql import DataFrame

# Get a list of all tables in the lakehouse
tables = spark.sql("SHOW TABLES").collect()

# Filter the list to only incllude the raw lap tables
lap_tables = [row['tableName'] for row in tables if row['tableName'].startswith('raw_laps_')]

df_list = []

# Loop through each table
for t in lap_tables:
    parts = t.split('_')

    year = parts[2]
    session = parts[-1]
    track = "_".join(parts[3:-1])

    df = spark.read.table(t)\
         .withColumn("Year", lit(year))\
         .withColumn("Track", lit(track))\
         .withColumn("Session", lit(session))

    df_list.append(df)

# Combine all dataframes
combined_raw_laps = reduce(DataFrame.unionByName, df_list)

# ====================================================================
# THE FIX: Standardize Team Names (Entity Resolution)
# If the team is "RB", permanently rename it to "Racing Bulls".
# ====================================================================
combined_raw_laps = combined_raw_laps.withColumn(
    "Team", 
    when(col("Team") == "RB", "Racing Bulls").otherwise(col("Team"))
)

# Save the master view
combined_raw_laps.createOrReplaceTempView("all_raw_laps")

print(f"Success! Combined {len(lap_tables)} raw lap tables into one master view and standardized Team Names.")

StatementMeta(, fc762db2-6a39-4d6a-a4cf-e254a1a8a939, 9, Finished, Available, Finished, False)

Success! Combined 224 raw lap tables into one master view and standardized Team Names.


In [8]:
# Use SQL to clean the data and keep ONLY perfectly accurate flying laps
silver_laps = spark.sql("""
    SELECT 
        Year,
        Track,
        Session,
        Team,
        CAST(DriverNumber AS INT) AS DriverNumber,
        CAST(CAST(LapNumber AS FLOAT) AS INT) AS LapNumber,
        LapTime,
        CAST(CAST(Stint AS FLOAT) AS INT) AS Stint,
        Compound AS TireCompound,
        CAST(TyreLife AS FLOAT) AS TireLifeLaps, 
        TrackStatus,
        IsAccurate
    FROM all_raw_laps
    WHERE LapTime != 'NaT' 
      AND LapTime != 'nan'
      AND LapTime IS NOT NULL
      AND IsAccurate = 'True' -- THIS is the bulletproof way to get clean flying laps!
""")

silver_laps.write.format("delta").mode("overwrite").saveAsTable("silver_all_laps")
print("Silver laps table successfully saved with NEW robust filtering!")

StatementMeta(, fc762db2-6a39-4d6a-a4cf-e254a1a8a939, 10, Finished, Available, Finished, False)

Silver laps table successfully saved with NEW robust filtering!


In [9]:
#Filter the list to Only include the raw weather table
weather_tables = [row['tableName'] for row in tables if row['tableName'].startswith('raw_weather_')]

weather_df_list = []

# Loop through each table
for t in weather_tables:
    parts = t.split('_')
    year = parts[2]
    session = parts[-1]
    track = "_".join(parts[3:-1])

    df = spark.read.table(t) \
        .withColumn("Year", lit(year))\
        .withColumn("Track", lit(track))\
        .withColumn("Session", lit(session))

    weather_df_list.append(df)

combined_raw_weather = reduce(DataFrame.unionByName, weather_df_list)

combined_raw_weather.createOrReplaceTempView("all_raw_weather")

print(f"Success! Combined {len(weather_tables)} raw weather tables into one master view")

StatementMeta(, fc762db2-6a39-4d6a-a4cf-e254a1a8a939, 11, Finished, Available, Finished, False)

Success! Combined 224 raw weather tables into one master view


In [10]:
# Use SQL to cast the weather metrics into proper numbers

silver_weather = spark.sql("""
    SELECT
        Year,
        Track,
        Session,
        Time,
        CAST(AirTemp AS FLOAT) AS AirTemp,
        CAST(TrackTemp AS FLOAT) AS TrackTemp,
        CAST(Humidity AS FLOAT) AS Humidity,
        CAST(WindSpeed AS FLOAT) AS WindSpeed,
        CAST(Rainfall as BOOLEAN) AS Rainfall
    FROM all_raw_weather
""")

silver_weather.write.format("delta").mode("overwrite").saveAsTable("silver_all_weather")

print("Silver weather table successfully saved!")

display(silver_weather)

StatementMeta(, fc762db2-6a39-4d6a-a4cf-e254a1a8a939, 12, Finished, Available, Finished, False)

Silver weather table successfully saved!


SynapseWidget(Synapse.DataFrame, 35860bc1-51d9-4c60-98d4-927ffe9bebdd)

In [11]:
# Diagnostic 1: Are the other teams inside the RAW data?
print("--- RAW LAPS BY TEAM ---")
raw_check = spark.sql("SELECT Team, COUNT(*) as RawLaps FROM all_raw_laps GROUP BY Team ORDER BY RawLaps DESC")
display(raw_check)

# Diagnostic 2: Did the other teams survive the SILVER filter?
print("--- CLEAN SILVER LAPS BY TEAM ---")
silver_check = spark.sql("SELECT Team, COUNT(*) as CleanLaps FROM silver_all_laps GROUP BY Team ORDER BY CleanLaps DESC")
display(silver_check)

StatementMeta(, fc762db2-6a39-4d6a-a4cf-e254a1a8a939, 13, Finished, Available, Finished, False)

--- RAW LAPS BY TEAM ---


SynapseWidget(Synapse.DataFrame, 89891d8b-29d5-4b93-b847-a6c7cde750d7)

--- CLEAN SILVER LAPS BY TEAM ---


SynapseWidget(Synapse.DataFrame, d7ad480e-02e2-426b-b5d4-e12f2fe78d24)

In [12]:
display(spark.sql("SELECT Team, COUNT(*) as RawLaps FROM all_raw_laps GROUP BY Team ORDER BY RawLaps DESC"))

StatementMeta(, fc762db2-6a39-4d6a-a4cf-e254a1a8a939, 14, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 8b39c837-7f1d-43d5-afb2-1603e4c93341)